# FinBERT Example Notebook

This notebooks shows how to train and use the FinBERT pre-trained language model for financial sentiment analysis.

## Modules 

In [1]:
from pathlib import Path
import shutil
import os
import logging
import sys
sys.path.append('..')

from textblob import TextBlob
from pprint import pprint
from sklearn.metrics import classification_report, balanced_accuracy_score

from transformers import AutoModelForSequenceClassification

from finbert.finbert import *
import finbert.utils as tools

%load_ext autoreload
%autoreload 2

project_dir = Path.cwd().parent
pd.set_option('max_colwidth', None)

In [2]:
logging.basicConfig(format = '%(asctime)s - %(levelname)s - %(name)s -   %(message)s',
                    datefmt = '%m/%d/%Y %H:%M:%S',
                    level = logging.ERROR)

## Prepare the model

### Setting path variables:
1. `lm_path`: the path for the pre-trained language model (If vanilla Bert is used then no need to set this one).
2. `cl_path`: the path where the classification model is saved.
3. `cl_data_path`: the path of the directory that contains the data files of `train.csv`, `validation.csv`, `test.csv`.
---

In the initialization of `bertmodel`, we can either use the original pre-trained weights from Google by giving `bm = 'bert-base-uncased`, or our further pre-trained language model by `bm = lm_path`


---
All of the configurations with the model is controlled with the `config` variable. 

In [3]:
import requests
import os
import shutil
from pathlib import Path
from tqdm import tqdm

# Download FinBERT language model
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
model_dir = project_root / 'models' / 'language_model' / 'finbertTRC2'
model_dir.mkdir(parents=True, exist_ok=True)

url = 'https://prosus-public.s3-eu-west-1.amazonaws.com/finbert/language-model/pytorch_model.bin'
dest_path = model_dir / 'pytorch_model.bin'

if not dest_path.exists():
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get('content-length', 0))
    with open(dest_path, 'wb') as f:
        with tqdm(total=total_size, unit='B', unit_scale=True, desc='Downloading') as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

# Copy config.json if it exists
config_source = project_root / 'config.json'
config_dest = model_dir / 'config.json'
if config_source.exists() and not config_dest.exists():
    shutil.copy2(str(config_source), str(config_dest))

In [4]:
# lm_path = project_dir / "models" / "language_model" / "finbertTRC2"
cl_path = project_dir / "models" / "classifier_model" / "finbert-sentiment"
cl_data_path = "NOSIBLE/financial-sentiment"
# cl_data_path = project_dir / "data" / "sentiment_data"

# cl_data_path = "zeroshot/twitter-financial-news-sentiment"

###  Configuring training parameters

You can find the explanations of the training parameters in the class docsctrings. 

In [5]:
# Clean the cl_path
try:
    shutil.rmtree(cl_path)
except:
    pass

# Load model with fallback to bert-base-uncased if local model not found
try:
    bertmodel = AutoModelForSequenceClassification.from_pretrained(
        str(lm_path), cache_dir=None, num_labels=3, local_files_only=True)
except:
    print("Local model not found, using bert-base-uncased as fallback")
    bertmodel = AutoModelForSequenceClassification.from_pretrained('FacebookAI/roberta-large-mnli', num_labels=3)

config = Config(data_dir=cl_data_path,
                bert_model=bertmodel,
                num_train_epochs=10,
                model_dir=cl_path,
                max_seq_length=128,
                train_batch_size=32,
                learning_rate=2e-5,
                output_mode='classification',
                warm_up_proportion=0.2,
                local_rank=-1,
                discriminate=True,
                gradual_unfreeze=True)

Local model not found, using bert-base-uncased as fallback


Some weights of the model checkpoint at FacebookAI/roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


`finbert` is our main class that encapsulates all the functionality. The list of class labels should be given in the prepare_model method call with label_list parameter.

In [6]:
finbert = FinBert(config)
finbert.base_model = 'bert-base-uncased'
finbert.config.discriminate=True
finbert.config.gradual_unfreeze=True

In [7]:
finbert.prepare_model(label_list=['positive','negative','neutral'])

12/24/2025 07:47:09 - INFO - finbert.finbert -   device: mps n_gpu: 1, distributed training: False, 16-bits training: False


## Fine-tune the model

In [8]:
# Get the training examples
train_data = finbert.get_data('train')

12/24/2025 07:47:11 - INFO - finbert.utils -   Dataset columns: ['text', 'label', 'netloc', 'url']
12/24/2025 07:47:12 - INFO - finbert.utils -   Loaded 100000 examples from Hugging Face dataset 'NOSIBLE/financial-sentiment' (split: train)
12/24/2025 07:47:21 - INFO - finbert.finbert -   Unique labels found in dataset: ['negative', 'neutral', 'positive']
12/24/2025 07:47:21 - INFO - finbert.finbert -   Expected labels: ['positive', 'negative', 'neutral']
12/24/2025 07:47:21 - INFO - finbert.finbert -   Label mapping: {'positive': {'count': 36257, 'normalized_to': 'positive'}, 'negative': {'count': 24434, 'normalized_to': 'negative'}, 'neutral': {'count': 39309, 'normalized_to': 'neutral'}}
12/24/2025 07:47:21 - INFO - finbert.finbert -   Normalized label distribution: {'neutral': 39309, 'positive': 36257, 'negative': 24434}
12/24/2025 07:47:21 - INFO - finbert.finbert -   Label 'positive': 36257 samples, weight: 2.7581
12/24/2025 07:47:21 - INFO - finbert.finbert -   Label 'negative': 

In [9]:
model = finbert.create_the_model()

### [Optional] Fine-tune only a subset of the model
The variable `freeze` determines the last layer (out of 12) to be freezed. You can skip this part if you want to fine-tune the whole model.

<span style="color:red">Important: </span>
Execute this step if you want a shorter training time in the expense of accuracy.

In [10]:
# This is for fine-tuning a subset of the model.

freeze = 6

base_model = None
# Detect model architecture (BERT vs RoBERTa)
if hasattr(model, 'bert'):
    base_model = model.bert
    model_type = 'bert'
elif hasattr(model, 'roberta'):
    base_model = model.roberta
    model_type = 'roberta'

for param in base_model.embeddings.parameters():
    param.requires_grad = False
    
for i in range(freeze):
    for param in base_model.encoder.layer[i].parameters():
        param.requires_grad = False

### Training

In [12]:
trained_model = finbert.train(train_examples=train_data, model=model)

12/24/2025 07:52:16 - INFO - finbert.utils -   Available splits: ['train']
12/24/2025 07:52:16 - WARNING - finbert.finbert -   Validation split not found. Splitting training data into train/validation (80/20 split).
12/24/2025 07:52:16 - INFO - finbert.finbert -   Split training data: 80000 train, 20000 validation
12/24/2025 07:52:16 - INFO - finbert.utils -   *** Example ***
12/24/2025 07:52:16 - INFO - finbert.utils -   guid: train-75221
12/24/2025 07:52:16 - INFO - finbert.utils -   tokens: [CLS] ga ##z ##pro ##m has since been working to rule , delivering basic contract volumes but not the usual season top - up flows . it booked almost no extra capacity for july , august , september , and now october . it has booked minimal amounts through the polish ya ##mal pipeline , another twist that is seriously alarm ##ing markets . ga ##z ##pro ##m says it has made up much of the short ##fall to europe with extra flows through the southern turk stream pipeline , which opened last year . whe

KeyboardInterrupt: 

In [ ]:
# plot the training loss
import matplotlib.pyplot as plt
import numpy as np

epochs = range(1, len(finbert.training_losses) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs, finbert.training_losses, 'b-o', label='Training Loss', linewidth=2, markersize=6)
plt.plot(epochs, finbert.validation_losses, 'r-s', label='Validation Loss', linewidth=2, markersize=6)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Validation Loss Over Epochs', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Training losses: {finbert.training_losses}")
print(f"Validation losses: {finbert.validation_losses}")
print(f"\nBest validation loss: {min(finbert.validation_losses):.4f} at epoch {np.argmin(finbert.validation_losses) + 1}")

## Test the model

`bert.evaluate` outputs the DataFrame, where true labels and logit values for each example is given

In [ ]:
test_data = finbert.get_data('test')

In [ ]:
results = finbert.evaluate(examples=test_data, model=trained_model)

### Prepare the classification report

In [ ]:
def report(df, cols=['label','prediction','logits']):
    df_valid = df[df[cols[0]] != 9090].copy()
    
    if len(df_valid) > 0:
        loss = CrossEntropyLoss(weight=finbert.class_weights)(
            torch.tensor(list(df_valid[cols[2]])),
            torch.tensor(list(df_valid[cols[0]]))
        )
        accuracy = (df_valid[cols[0]] == df_valid[cols[1]]).mean()
        balanced_acc = balanced_accuracy_score(df_valid[cols[0]], df_valid[cols[1]])
        print(f"Loss: {loss:.5f}")
        print(f"Accuracy: {accuracy:.5f}")
        print(f"Balanced Accuracy: {balanced_acc:.5f}")
        print("\nClassification Report (with labels):")
        print(classification_report(df_valid[cols[0]], df_valid[cols[1]]))
    else:
        print("No ground truth labels available. Showing predictions only.")
    
    label_names = ['positive', 'negative', 'neutral']
    print("\nPrediction Distribution:")
    for idx, count in df[cols[1]].value_counts().sort_index().items():
        name = label_names[int(idx)] if int(idx) < len(label_names) else f'class_{idx}'
        print(f'  {name}: {count} ({count/len(df)*100:.1f}%)')


In [ ]:
results['prediction'] = results.predictions.apply(lambda x: np.argmax(x,axis=0))

In [ ]:
report(results,cols=['labels','prediction','predictions'])

### Evaluate on Validation Set (with labels)

Evaluate the model on validation data which has ground truth labels to get metrics.

In [ ]:
# Get validation data (which has labels)
validation_data = finbert.get_data("validation")

# Evaluate on validation set
val_results = finbert.evaluate(examples=validation_data, model=trained_model)
val_results["prediction"] = val_results.predictions.apply(lambda x: np.argmax(x,axis=0))

# Generate full report with metrics
report(val_results, cols=["labels","prediction","predictions"])

### Get predictions

With the `predict` function, given a piece of text, we split it into a list of sentences and then predict sentiment for each sentence. The output is written into a dataframe. Predictions are represented in three different columns: 

1) `logit`: probabilities for each class

2) `prediction`: predicted label

3) `sentiment_score`: sentiment score calculated as: probability of positive - probability of negative

Below we analyze a paragraph taken out of [this](https://www.economist.com/finance-and-economics/2019/01/03/a-profit-warning-from-apple-jolts-markets) article from The Economist. For comparison purposes, we also put the sentiments predicted with TextBlob.
> Later that day Apple said it was revising down its earnings expectations in the fourth quarter of 2018, largely because of lower sales and signs of economic weakness in China. The news rapidly infected financial markets. Apple’s share price fell by around 7% in after-hours trading and the decline was extended to more than 10% when the market opened. The dollar fell by 3.7% against the yen in a matter of minutes after the announcement, before rapidly recovering some ground. Asian stockmarkets closed down on January 3rd and European ones opened lower. Yields on government bonds fell as investors fled to the traditional haven in a market storm.

In [ ]:
text = "Later that day Apple said it was revising down its earnings expectations in \
the fourth quarter of 2018, largely because of lower sales and signs of economic weakness in China. \
The news rapidly infected financial markets. Apple’s share price fell by around 7% in after-hours \
trading and the decline was extended to more than 10% when the market opened. The dollar fell \
by 3.7% against the yen in a matter of minutes after the announcement, before rapidly recovering \
some ground. Asian stockmarkets closed down on January 3rd and European ones opened lower. \
Yields on government bonds fell as investors fled to the traditional haven in a market storm."

In [ ]:
cl_path = project_dir/'models'/'classifier_model'/'finbert-sentiment'
cl_path_str = os.path.abspath(str(cl_path))

# Check if the model path exists
if not os.path.exists(cl_path_str):
    raise FileNotFoundError(f"Classifier model path does not exist: {cl_path_str}. Please train the model first or ensure the path is correct.")
else:
    model = AutoModelForSequenceClassification.from_pretrained(cl_path_str, cache_dir=None, num_labels=3, local_files_only=True)

In [ ]:
import nltk
nltk.download('punkt')

In [ ]:
result = predict(text,model)

In [ ]:
blob = TextBlob(text)
result['textblob_prediction'] = [sentence.sentiment.polarity for sentence in blob.sentences]
result

In [ ]:
print(f'Average sentiment is %.5f.' % (result.sentiment_score.mean()))

Here is another example

In [ ]:
text2 = "Shares in the spin-off of South African e-commerce group Naspers surged more than 25% \
in the first minutes of their market debut in Amsterdam on Wednesday. Bob van Dijk, CEO of \
Naspers and Prosus Group poses at Amsterdam's stock exchange, as Prosus begins trading on the \
Euronext stock exchange in Amsterdam, Netherlands, September 11, 2019. REUTERS/Piroschka van de Wouw \
Prosus comprises Naspers’ global empire of consumer internet assets, with the jewel in the crown a \
31% stake in Chinese tech titan Tencent. There is 'way more demand than is even available, so that’s \
good,' said the CEO of Euronext Amsterdam, Maurice van Tilburg. 'It’s going to be an interesting \
hour of trade after opening this morning.' Euronext had given an indicative price of 58.70 euros \
per share for Prosus, implying a market value of 95.3 billion euros ($105 billion). The shares \
jumped to 76 euros on opening and were trading at 75 euros at 0719 GMT."

In [ ]:
result2 = predict(text2,model)
blob = TextBlob(text2)
result2['textblob_prediction'] = [sentence.sentiment.polarity for sentence in blob.sentences]

In [ ]:
result2

In [ ]:
print(f'Average sentiment is %.5f.' % (result2.sentiment_score.mean()))